In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import r2_score, accuracy_score
import pickle


In [2]:
df = pd.read_csv('Data_Sheet/earthquake_1995-2023.csv')
df.head()


,title,magnitude,date_time,cdi,mmi,alert,tsunami,sig,net,nst,dmin,gap,magType,depth,latitude,longitude,location,continent,country
0,"M 6.5 - 42 km W of Sola, Vanuatu",6.5,16-08-2023 12:47,7,4,green,0,657,us,114,7.177000,25.0,mww,192.955,-13.8814,167.1580,"Sola, Vanuatu",NaN,Vanuatu
1,"M 6.5 - 43 km S of Intipucá, El Salvador",6.5,19-07-2023 00:22,8,6,yellow,0,775,us,92,0.679000,40.0,mww,69.727,12.8140,-88.1265,"Intipucá, El Salvador",NaN,NaN
2,"M 6.6 - 25 km ESE of Loncopué, Argentina",6.6,17-07-2023 03:05,7,5,green,0,899,us,70,1.634000,28.0,mww,171.371,-38.1911,-70.3731,"Loncopué, Argentina",South America,Argentina
3,"M 7.2 - 98 km S of Sand Point, Alaska",7.2,16-07-2023 06:48,6,6,green,1,860,us,173,0.907000,36.0,mww,32.571,54.3844,-160.6990,"Sand Point, Alaska",NaN,NaN
4,M 7.3 - Alaska Peninsula,7.3,16-07-2023 06:48,0,5,NaN,1,820,at,79,0.879451,172.8,Mi,21.000,54.4900,-160.7960,Alaska Peninsula,NaN,NaN


In [3]:
#Null values 

df.isna().sum()

title          0
magnitude      0
date_time      0
cdi            0
mmi            0
alert        551
tsunami        0
sig            0
net            0
nst            0
dmin           0
gap            0
magType        0
depth          0
latitude       0
longitude      0
location       6
continent    716
country      349
dtype: int64

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 19 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   title      1000 non-null   object 
 1   magnitude  1000 non-null   float64
 2   date_time  1000 non-null   object 
 3   cdi        1000 non-null   int64  
 4   mmi        1000 non-null   int64  
 5   alert      449 non-null    object 
 6   tsunami    1000 non-null   int64  
 7   sig        1000 non-null   int64  
 8   net        1000 non-null   object 
 9   nst        1000 non-null   int64  
 10  dmin       1000 non-null   float64
 11  gap        1000 non-null   float64
 12  magType    1000 non-null   object 
 13  depth      1000 non-null   float64
 14  latitude   1000 non-null   float64
 15  longitude  1000 non-null   float64
 16  location   994 non-null    object 
 17  continent  284 non-null    object 
 18  country    651 non-null    object 
dtypes: float64(6), int64(5), object(8)
memory usage: 

In [ ]:
#Fillup the null values
df['alert'] = df['alert'].fillna(df['alert'].mode()[0])
df["location"] = df["location"].fillna("Unknown")
df['location'] = df['location'].fillna('Unknown')
df['country'] = df['country'].fillna('Unknown')

df['dmin'] = round(df['dmin'],3)
df['depth'] = round(df['depth'],2)

In [ ]:
#Do saperate the Date and time values.

df['date_time'] = pd.to_datetime(df['date_time'], errors='coerce')

df['year'] = df['date_time'].dt.year
df['month'] = df['date_time'].dt.month
df['day'] = df['date_time'].dt.day
df['hour'] = df['date_time'].dt.hour
df['minute'] = df['date_time'].dt.minute

df.drop(columns=['date_time'], inplace=True)


C:\Users\Krushn\AppData\Local\Temp\ipykernel_12900\3599483473.py:1: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['date_time'] = pd.to_datetime(df['date_time'], errors='coerce')


In [ ]:
categorical_columns = ['title','alert','net','magType','continent','country','location','tsunami']

encoders = {}

for col in categorical_columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

In [8]:
features = [
    'title',    'cdi',    'mmi',    'net',    'nst',    'dmin',
    'gap',    'magType',    'depth',    'latitude',    'longitude',
    'location',    'continent',    'country',    'year',    'month',
    'day',    'hour',    'minute' , 'tsunami'
]

X = df[features]

In [19]:
print("Training Magnitude Prediction Model...\n")

# Features and Target
X_magnitude = X
y_magnitude = df["magnitude"]

# Train Test Split
X_train_magnitude, X_test_magnitude, y_train_magnitude, y_test_magnitude = train_test_split(
    X_magnitude, y_magnitude,
    test_size=0.2,
    random_state=42
)

# Model
magnitude_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

# Train
magnitude_model.fit(X_train_magnitude, y_train_magnitude)

# Prediction
y_pred_magnitude = magnitude_model.predict(X_test_magnitude)

# Evaluation
score_magnitude = r2_score(y_test_magnitude, y_pred_magnitude)

print(f"R² Score : {score_magnitude}")

Training Magnitude Prediction Model...

R² Score : 0.9987950345417066


In [20]:
print("Training Significance Prediction Model...\n")

# Features and Target
X_sig = X
y_sig = df['sig']

# Train Test Split
X_train_sig, X_test_sig, y_train_sig, y_test_sig = train_test_split(
    X_sig, y_sig,
    test_size=0.2,
    random_state=42
)

# Model
sig_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

# Train
sig_model.fit(X_train_sig, y_train_sig)

# Prediction
y_pred = sig_model.predict(X_test_sig)

# Evaluation
score_sig = r2_score(y_test_sig, y_pred)

print(f"R² Score : {score_sig}")

Training Significance Prediction Model...

R² Score : 0.6094792941332885


In [21]:
print("Training Alert Prediction Model...\n")

# Features and Target
X_alert = X
y_alert = df['alert']

# Train Test Split
X_train_alert, X_test_alert, y_train_alert, y_test_alert = train_test_split(
    X_alert, y_alert,
    test_size=0.2,
    random_state=42
)

# Model
alert_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Train
alert_model.fit(X_train_alert, y_train_alert)

# Prediction
y_pred_alert = alert_model.predict(X_test_alert)

# Evaluation
score_alert = accuracy_score(y_test_alert, y_pred_alert)

print(f"Accuracy : {score_alert}")

Training Alert Prediction Model...

Accuracy : 0.92


In [22]:
print("Training Tsunami Prediction Model...\n")

# Features and Target
X_tsunami = X
y_tsunami = df['tsunami']

# Train Test Split
X_train_tsunami, X_test_tsunami, y_train_tsunami, y_test_tsunami = train_test_split(
    X_tsunami, y_tsunami,
    test_size=0.2,
    random_state=42
)

# Model
tsunami_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Train
tsunami_model.fit(X_train_tsunami, y_train_tsunami)

# Prediction
y_pred_tsunami = tsunami_model.predict(X_test_tsunami)

# Evaluation
score_tsunami = accuracy_score(y_test_tsunami, y_pred_tsunami)

print(f"Accuracy : {score_tsunami}")

Training Tsunami Prediction Model...

Accuracy : 1.0


In [23]:
models = {
    "magnitude_model": magnitude_model,
    "significance_model": sig_model,
    "alert_model": alert_model,
    "tsunami_model": tsunami_model,
    "encoders": encoders
}

with open("Models/earthquake_models.pkl", "wb") as file:
    pickle.dump(models, file)

print("Model is created into the 'Models/' folder...")

Model is created into the 'Models/' folder...


In [24]:
# User Inputs

title_value = "M 7.8 - Central Turkey"

continent_value = "Asia"
country_value = "Turkiye"
location_value = "Central Turkey"

year = 2023
month = 2
day = 6
hour = 1
minute = 17

latitude = 37.2012
longitude = 36.9997

depth = 17.925

magType_value = "mww"

cdi = 9
mmi = 9

net_value = "us"

nst = 118

dmin = 1.919

gap = 32
nst = 168

dmin = 0.345

gap = 24.8

tsunami_value = 0  # 0 = No, 1 = Yes

print("User input completed....")

User input completed....


In [25]:
# Encode the categorical columns using the fitted encoders from training

title = encoders["title"].transform([title_value])[0]
continent = encoders["continent"].transform([continent_value])[0]
country = encoders["country"].transform([country_value])[0]
location = encoders["location"].transform([location_value])[0]
magType = encoders["magType"].transform([magType_value])[0]
net = encoders["net"].transform([net_value])[0]
tsunami = encoders["tsunami"].transform([str(tsunami_value)])[0]

print("User input Encoding is completed.")

User input Encoding is completed.


In [26]:
# Dataframe of the user input

user_data = pd.DataFrame({
    "title":[title],
    "cdi":[cdi],
    "mmi":[mmi],
    "net":[net],
    "nst":[nst],
    "dmin":[dmin],
    "gap":[gap],
    "magType":[magType],
    "depth":[depth],
    "latitude":[latitude],
    "longitude":[longitude],
    "location":[location],
    "continent":[continent],
    "country":[country],
    "year":[year],
    "month":[month],
    "day":[day],
    "hour":[hour],
    "minute":[minute],
    "tsunami":[tsunami]
})

#Prediction of multiple models

predicted_magnitude = magnitude_model.predict(user_data)[0]
predicted_sig = sig_model.predict(user_data)[0]
predicted_alert = alert_model.predict(user_data)[0]
predicted_tsunami = tsunami_model.predict(user_data)[0]

predicted_alert = encoders["alert"].inverse_transform([predicted_alert])[0]

predicted_tsunami = encoders["tsunami"].inverse_transform([predicted_tsunami])[0]

In [27]:
print("\n" + "=" * 50)
print("      EARTHQUAKE PREDICTION REPORT")
print("=" * 50)

print(f"Predicted Magnitude     : {predicted_magnitude:.2f}")
print(f"Predicted Significance  : {predicted_sig:.2f}")
print(f"Predicted Alert Level   : {predicted_alert}")
print(f"Predicted Tsunami       : {'Yes' if predicted_tsunami == 1 else 'No'}")

print("=" * 50)


      EARTHQUAKE PREDICTION REPORT
Predicted Magnitude     : 7.80
Predicted Significance  : 2198.05
Predicted Alert Level   : red
Predicted Tsunami       : No
